# ROADNETRA AI — Model A: Infrastructure & Defect Detection Training
**Problem Statement 03.1: Vision in the Wild (IEEE Hackathon 2026)**  
**Team ESPADA — Bhawesh Kumar Gautam**  

---
### Instructions:
1. In the menu bar above, go to **Runtime** → **Change runtime type** → select **T4 GPU**.
2. Upload your `unified_infrastructure.zip` to Colab (or mount Google Drive).
3. Run the cells below sequentially.

In [ ]:
# Step 1: Install Ultralytics and check GPU
!pip install -q ultralytics

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Training on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not detected. Go to Runtime -> Change runtime type -> Select T4 GPU.")

In [ ]:
# Step 2: Extract the Unified Infrastructure Dataset
!unzip -q /content/unified_infrastructure.zip -d /content/dataset
print("Dataset extracted successfully to /content/dataset/")

In [ ]:
# Step 3: Configure dataset path for Colab environment
import yaml

yaml_path = '/content/dataset/data.yaml'
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['path'] = '/content/dataset'

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)

print("Config verified:")
print(cfg)

In [ ]:
# Step 4: Execute Fine-Tuning with 'Vision in the Wild' Augmentation Pipeline
from ultralytics import YOLO

# Load base pre-trained checkpoint
model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=40,
    imgsz=640,
    batch=16,
    device=0,               # NVIDIA Cloud GPU
    patience=10,            # Early stopping if loss plateaus
    
    # --- 'Vision in the Wild' Augmentation Hyperparameters ---
    hsv_h=0.015,            # Subtle hue shift
    hsv_s=0.7,              # Saturation shift (wet tarmac, rainwater, smog)
    hsv_v=0.4,              # Brightness jitter (night CCTV, tunnels, sun glare)
    degrees=10.0,           # Pole sway & camera tilt
    translate=0.1,          # Boundary frame movement
    scale=0.5,              # Distant vs close hazard scaling
    perspective=0.0005,     # Highway vanishing lines
    mosaic=1.0,             # Occlusion & multi-hazard recognition
    mixup=0.15,             # Blends background noise into scene
    # ---------------------------------------------------------
    
    project='roadnetra_runs',
    name='infra_defect_v1',
    exist_ok=True,
    plots=True
)

In [ ]:
# Step 5: Validate and evaluate performance
metrics = model.val()
print(f"Validation mAP@50: {metrics.box.map50:.4f}")
print(f"Validation mAP@50-95: {metrics.box.map:.4f}")

In [ ]:
# Step 6: Download the trained best.pt weights to your laptop
from google.colab import files

weights_path = '/content/roadnetra_runs/infra_defect_v1/weights/best.pt'
files.download(weights_path)
print("Downloaded best.pt. Rename it to 'pothole_best.pt' and place in your ieee/models/ folder!")